In [1]:
import os
import glob
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import(
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

MODEL = "distilbert-base-uncased"
SEED = 42


In [2]:
path = os.path.join(os.getcwd(), '*.csv')
path = list(glob.glob(path))
print(path)

['d:\\Project\\Transformers\\Ag_News\\test.csv', 'd:\\Project\\Transformers\\Ag_News\\train.csv']


In [11]:
df1 = pd.read_csv(path[0])
df2 = pd.read_csv(path[1])
df = pd.concat([df1, df2], axis=0)

print(df.shape)

df.head()

(127600, 3)


,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [12]:
df["Text"] = df["Title"].fillna("") + ". " + df["Description"].fillna("")
df = df.drop(columns=['Description', 'Title'])

In [13]:
df = df.rename(columns={'Class Index':'label', 'Description':'text'})

In [14]:
df.head()

,label,Text
0,3,Fears for T N pension after talks. Unions repr...
1,4,The Race is On: Second Private Team Sets Launc...
2,4,Ky. Company Wins Grant to Study Peptides (AP)....
3,4,Prediction Unit Helps Forecast Wildfires (AP)....
4,4,Calif. Aims to Limit Farm-Related Smog (AP). A...


In [15]:
# 1-World, 2-Sports, 3-Business, 4-Sci/Tech
df['label'] = df['label'].replace({1:'world', 2:'sports', 3:'business', 4:'Sci/Tech'})
labels = sorted(df['label'].unique().tolist(), key=str)
label2id = {l : i for i, l in enumerate(labels)}
id2label = {i : str(l) for l, i in label2id.items()}

df['label'] = df['label'].map(label2id)

In [16]:
ds = Dataset.from_pandas(df, preserve_index=False).train_test_split(test_size=0.3, seed=SEED)

In [18]:
tok = AutoTokenizer.from_pretrained(
    MODEL
)

ds["train"] = ds["train"].shuffle(seed=SEED).select(range(20000))
ds["test"]  = ds["test"].shuffle(seed=SEED).select(range(4000))

ds = ds.map(lambda b: tok(b['Text'], truncation=False, max_length=128), batched=True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=4, id2label=id2label, label2id=label2id
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': (preds == p.label_ids).mean()}

In [21]:
args = TrainingArguments(
    output_dir='out',
    per_device_train_batch_size=32,
    num_train_epochs=2,
    report_to='none',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=3e-5
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tok)
)

In [22]:
trainer.train()
print(trainer.evaluate())

  0%|          | 0/1250 [00:00<?, ?it/s]

{'loss': 0.3448, 'grad_norm': 4.208082675933838, 'learning_rate': 1.8e-05, 'epoch': 0.8}


  0%|          | 0/500 [00:00<?, ?it/s]

{'eval_loss': 0.23018990457057953, 'eval_accuracy': 0.919, 'eval_runtime': 38.2507, 'eval_samples_per_second': 104.573, 'eval_steps_per_second': 13.072, 'epoch': 1.0}
{'loss': 0.1912, 'grad_norm': 4.175643444061279, 'learning_rate': 6e-06, 'epoch': 1.6}


  0%|          | 0/500 [00:00<?, ?it/s]

{'eval_loss': 0.22631004452705383, 'eval_accuracy': 0.92725, 'eval_runtime': 39.0843, 'eval_samples_per_second': 102.343, 'eval_steps_per_second': 12.793, 'epoch': 2.0}
{'train_runtime': 1929.5855, 'train_samples_per_second': 20.73, 'train_steps_per_second': 0.648, 'train_loss': 0.24683171997070313, 'epoch': 2.0}


  0%|          | 0/500 [00:00<?, ?it/s]

{'eval_loss': 0.22631004452705383, 'eval_accuracy': 0.92725, 'eval_runtime': 39.3901, 'eval_samples_per_second': 101.548, 'eval_steps_per_second': 12.694, 'epoch': 2.0}


In [23]:
trainer.save_model('New_mymodel')
tok.save_pretrained('New_mymodel')

('New_mymodel\\tokenizer_config.json',
 'New_mymodel\\special_tokens_map.json',
 'New_mymodel\\vocab.txt',
 'New_mymodel\\added_tokens.json',
 'New_mymodel\\tokenizer.json')

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(ds["test"])
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids
names = [id2label[i] for i in range(len(id2label))]

print(id2label)
print(classification_report(y_true, y_pred, target_names=names))
print(confusion_matrix(y_true, y_pred))

  0%|          | 0/500 [00:00<?, ?it/s]

{0: 'Sci/Tech', 1: 'business', 2: 'sports', 3: 'world'}
              precision    recall  f1-score   support

    Sci/Tech       0.88      0.92      0.90      1025
    business       0.91      0.87      0.89       999
      sports       0.98      0.98      0.98      1013
       world       0.94      0.93      0.94       963

    accuracy                           0.93      4000
   macro avg       0.93      0.93      0.93      4000
weighted avg       0.93      0.93      0.93      4000

[[948  55   2  20]
 [ 98 867   6  28]
 [  5   5 995   8]
 [ 27  22  15 899]]
